# SpaceX Falcon 9 First Stage Landing Prediction

## Interactive Visual Analytics with Folium
This notebook explores the geographic context of Falcon 9 launches using interactive maps.

## Contents
1. Load and prepare launch data
2. Map launch sites and landing outcomes
3. Perform proximity analysis with coastline and city references
4. Summarize geographic insights relevant to landing operations

## 1. Import Required Libraries
Import mapping and data tools used throughout the notebook.

In [1]:
import folium
import pandas as pd
from folium.features import DivIcon
from folium.plugins import MarkerCluster, MousePosition

In [2]:
spacex_df = pd.read_csv("../data/processed/spacex_launch_data_clean.csv")

## 2. Load and Prepare Launch Data
Load the cleaned dataset, remove incomplete labels, and standardize key column types.

In [3]:
spacex_df = pd.read_csv('../data/processed/spacex_launch_data_clean.csv')

spacex_df = spacex_df[spacex_df['Class'].notna()].copy()
spacex_df['Class'] = spacex_df['Class'].astype(int)
spacex_df['LaunchSite'] = spacex_df['LaunchSite'].astype(str)
spacex_df['Latitude'] = spacex_df['Latitude'].astype(float)
spacex_df['Longitude'] = spacex_df['Longitude'].astype(float)
spacex_df['BoosterVersion'] = spacex_df['BoosterVersion'].astype(str)
spacex_df['PayloadMass'] = spacex_df['PayloadMass'].astype(float)
spacex_df['Orbit'] = spacex_df['Orbit'].astype(str)

spacex_df['Date'] = pd.to_datetime(spacex_df['Date'])
spacex_df['Year'] = spacex_df['Date'].dt.year

print(f"Total launches: {len(spacex_df)}")
print(f"Launch sites: {spacex_df['LaunchSite'].nunique()}")
print(f"Date range: {spacex_df['Year'].min()} - {spacex_df['Year'].max()}")

Total launches: 90
Launch sites: 3
Date range: 2010 - 2020


## 3. Build Launch Site Summary
Create a table of unique launch sites and their coordinates for map annotation.

In [4]:
# Select relevant columns for geographic analysis
spacex_df = spacex_df[['LaunchSite', 'Latitude', 'Longitude', 'Class']]

# Create a summary dataframe with unique launch sites and their coordinates
# Group by LaunchSite and take the first occurrence to get unique site coordinates
launch_sites_df = spacex_df.groupby(['LaunchSite'], as_index=False).first()
launch_sites_df = launch_sites_df[['LaunchSite', 'Latitude', 'Longitude']]

# Display the launch sites
print(f"Number of unique launch sites: {len(launch_sites_df)}")
launch_sites_df

Number of unique launch sites: 3


,LaunchSite,Latitude,Longitude
0,CCSFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


## 4. Create a Base Map
### 4.1 Initialize Map at NASA Johnson Space Center

In [5]:
# Define NASA Johnson Space Center coordinates [Latitude, Longitude]
# Location: Houston, Texas
nasa_coordinate = [29.559684888503615, -95.0830971930759]

# Create a Folium map centered at NASA JSC with appropriate zoom level
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

site_map

### 4.2 Highlight NASA JSC with Circle and Label

In [6]:
# Create a circle around NASA Johnson Space Center
# - radius: 1000 meters (1 km)
# - color: Orange (#d35400) to make it stand out
# - fill: True to fill the circle area
circle = folium.Circle(
    nasa_coordinate, 
    radius=1000, 
    color='#d35400', 
    fill=True
).add_child(folium.Popup('NASA Johnson Space Center'))

# Create a custom text label marker for NASA JSC
# Using DivIcon to create an HTML-based label instead of a standard icon
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12pt; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
    )
)

# Add both circle and marker to the map
site_map.add_child(circle)
site_map.add_child(marker)

site_map

## 5. Map All SpaceX Launch Sites
### 5.1 Add Launch Site Circles and Labels

In [7]:
# Initialize a new map with a wider zoom to show all launch sites
# Using NASA JSC as the center point with zoom_start=5 for broader view
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Iterate through each launch site and add visual markers
for index, row in launch_sites_df.iterrows():
    # Create a circle for each launch site
    circle = folium.Circle(
        [row['Latitude'], row['Longitude']], 
        radius=1000,  # 1 km radius
        color='#d35400',  # Orange color for visibility
        fill=True
    ).add_child(folium.Popup(row['LaunchSite']))
    
    # Create a text label marker for the launch site name
    marker = folium.map.Marker(
        [row['Latitude'], row['Longitude']],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12pt; color:#d35400;"><b>%s</b></div>' % row['LaunchSite'],
        )
    )
    
    # Add circle and marker to the map
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

## 6. Visualize Launch Success and Failure
### 6.1 Add Clustered Outcome Markers
Use green markers for successful landings and red markers for failed landings.

In [8]:
# Create a MarkerCluster to group nearby launch markers
# This improves visualization when multiple launches occur at the same site
marker_cluster = MarkerCluster().add_to(site_map)

# Add a marker for each launch, color-coded by success/failure
for index, row in spacex_df.iterrows():
    # Determine marker color based on launch outcome
    # Class = 1: Successful landing (green)
    # Class = 0: Failed landing (red)
    if row['Class'] == 1:
        color = 'green'
    else:
        color = 'red'
    
    # Create and add marker to the cluster
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color=color),
    ).add_to(marker_cluster)

site_map

### 6.2 Inspect Recent Launch Records

In [9]:
# Display the last 10 launches to check the data
spacex_df.tail(10)

,LaunchSite,Latitude,Longitude,Class
80,CCSFS SLC 40,28.561857,-80.577366,1
81,CCSFS SLC 40,28.561857,-80.577366,1
82,CCSFS SLC 40,28.561857,-80.577366,1
83,CCSFS SLC 40,28.561857,-80.577366,1
84,CCSFS SLC 40,28.561857,-80.577366,1
85,KSC LC 39A,28.608058,-80.603956,1
86,KSC LC 39A,28.608058,-80.603956,1
87,KSC LC 39A,28.608058,-80.603956,1
88,CCSFS SLC 40,28.561857,-80.577366,1
89,CCSFS SLC 40,28.561857,-80.577366,1


### 6.3 Alternative Marker Coloring Approach

In [10]:
# Create a new MarkerCluster for fresh visualization
marker_cluster = MarkerCluster()

# Create a new column 'marker_color' in spacex_df to store marker colors
# This makes the code cleaner by pre-computing colors
# Green = Successful landing, Red = Failed landing
spacex_df['marker_color'] = spacex_df['Class'].apply(lambda x: 'green' if x == 1 else 'red') 

# Add markers using the pre-computed color column
for index, row in spacex_df.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color=row['marker_color']),
    ).add_to(marker_cluster)

# Add the marker cluster to the map
site_map.add_child(marker_cluster)

site_map

### 6.4 Custom Marker Rendering

In [11]:
# Create a new fresh map for custom marker demonstration
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add launch site circles and labels
for index, row in launch_sites_df.iterrows():
    circle = folium.Circle(
        [row['Latitude'], row['Longitude']], 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(row['LaunchSite']))
    
    marker = folium.map.Marker(
        [row['Latitude'], row['Longitude']],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12pt; color:#d35400;"><b>%s</b></div>' % row['LaunchSite'],
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

# Add marker cluster for launches
marker_cluster = MarkerCluster()

# Create markers for each launch with custom styling
for index, record in spacex_df.iterrows():
    # Create marker with color based on success/failure
    marker = folium.Marker(
        location=[record['Latitude'], record['Longitude']],
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

# Add the completed marker cluster to the map
site_map.add_child(marker_cluster)

site_map

## 7. Proximity Analysis
### 7.1 Build Coastline Reference Points

In [12]:
# Create coastline reference data for proximity calculations
# These coordinates represent approximate coastline points near the launch sites

# CCSFS SLC 40 is in Cape Canaveral, Florida (Atlantic coast)
# VAFB SLC 4E is in Vandenberg, California (Pacific coast)
coastline_df = pd.DataFrame({
    'Name': ['Cape Canaveral Coast', 'Vandenberg Coast', 'Florida East Coast', 'California Central Coast'],
    'Latitude': [28.56342, 34.6324, 28.6, 34.7],
    'Longitude': [-80.567, -120.6, -80.5, -120.5]
})

print("Coastline reference points:")
coastline_df

Coastline reference points:


,Name,Latitude,Longitude
0,Cape Canaveral Coast,28.56342,-80.567
1,Vandenberg Coast,34.63240,-120.600
2,Florida East Coast,28.60000,-80.500
3,California Central Coast,34.70000,-120.500


### 7.2 Distance Function (Haversine Formula)

In [13]:
# Import mathematical functions for distance calculation
from math import sin, cos, sqrt, atan2, radians 

def calculate_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on Earth using the Haversine formula.
    
    Parameters:
    -----------
    lat1, lon1 : float
        Latitude and longitude of point 1 in degrees
    lat2, lon2 : float
        Latitude and longitude of point 2 in degrees
    
    Returns:
    --------
    float
        Distance in kilometers
    """
    # Approximate radius of Earth in kilometers
    R = 6373.0

    # Convert coordinates from degrees to radians
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    # Calculate differences in coordinates
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Haversine formula
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    # Calculate distance
    distance = R * c

    return distance

# Calculate and store the distance to the closest coastline for each launch site
launch_sites_df['DistanceToCoastline'] = 0.0    

for index, launch_site in launch_sites_df.iterrows():
    # Initialize variables to track the closest coastline point
    closest_coastline = None
    min_distance = float('inf')
    
    # Find the closest coastline point by checking all coastline coordinates
    for _, coastline in coastline_df.iterrows():
        distance = calculate_distance(
            launch_site['Latitude'], 
            launch_site['Longitude'], 
            coastline['Latitude'], 
            coastline['Longitude']
        )
        
        if distance < min_distance:
            min_distance = distance
            closest_coastline = coastline
    
    # Store the minimum distance to the closest coastline
    launch_sites_df.at[index, 'DistanceToCoastline'] = min_distance

# Display results
print("Launch Sites with Distance to Coastline:")
launch_sites_df

Launch Sites with Distance to Coastline:


,LaunchSite,Latitude,Longitude,DistanceToCoastline
0,CCSFS SLC 40,28.561857,-80.577366,1.027494
1,KSC LC 39A,28.608058,-80.603956,6.138497
2,VAFB SLC 4E,34.632093,-120.610829,0.991677


### 7.3 Annotate Closest Coastline Distances

In [14]:
# For each launch site, find and mark the closest coastline point
for index, launch_site in launch_sites_df.iterrows():
    # Find the closest coastline point
    closest_coastline = None
    min_distance = float('inf')
    
    for _, coastline in coastline_df.iterrows():
        distance = calculate_distance(
            launch_site['Latitude'], 
            launch_site['Longitude'], 
            coastline['Latitude'], 
            coastline['Longitude']
        )
        
        if distance < min_distance:
            min_distance = distance
            closest_coastline = coastline
    
    # Create a marker at the closest coastline point showing the distance
    marker = folium.Marker(
        location=[closest_coastline['Latitude'], closest_coastline['Longitude']],
        icon=DivIcon(
            icon_size=(200, 36),
            icon_anchor=(0, 0),
            html='<div style="font-size: 10pt; color:#d35400;"><b>Distance: %.2f km</b></div>' % min_distance,
        )
    )
    site_map.add_child(marker)

site_map

### 7.4 Draw Site-to-Coastline Segments

In [15]:
# Draw lines connecting launch sites to their closest coastline points
for index, launch_site in launch_sites_df.iterrows():
    # Find the closest coastline point
    closest_coastline = None
    min_distance = float('inf')
    
    for _, coastline in coastline_df.iterrows():
        distance = calculate_distance(
            launch_site['Latitude'], 
            launch_site['Longitude'], 
            coastline['Latitude'], 
            coastline['Longitude']
        )
        
        if distance < min_distance:
            min_distance = distance
            closest_coastline = coastline
    
    # Create a line (PolyLine) connecting the launch site to the coastline
    coordinates = [
        [launch_site['Latitude'], launch_site['Longitude']], 
        [closest_coastline['Latitude'], closest_coastline['Longitude']]
    ]
    lines = folium.PolyLine(locations=coordinates, weight=2, color='blue', opacity=0.6)
    site_map.add_child(lines)

site_map

## 8. Interactive Map Tools
### 8.1 Add Mouse Position Coordinates

In [16]:
# Add a MousePosition plugin to display coordinates
# This allows you to hover over any point on the map and see its coordinates
# Useful for identifying locations of cities, railways, highways, etc.
mouse_position = MousePosition(
    position='topright',         # Position on the map
    separator=' Long: ',         # Separator between lat and long
    empty_string='NaN',          # Display when no position
    lng_first=False,             # Show latitude first
    num_digits=20,               # Number of decimal places
    prefix='Lat:',               # Prefix for coordinates
)

site_map.add_child(mouse_position)

site_map

### 8.2 Mark Nearby Cities and Distances

In [17]:
# Define nearby cities for each launch site
# You can use the MousePosition tool to find accurate coordinates
# Cape Canaveral: Nearby city is Cape Canaveral/Cocoa Beach
# Vandenberg: Nearby city is Lompoc, CA

cities_df = pd.DataFrame({
    'Name': ['Cape Canaveral', 'Lompoc'],
    'Latitude': [28.485833, 34.6391],
    'Longitude': [-80.544444, -120.4579]
})

# For each launch site, mark the nearest city and draw a line
for index, launch_site in launch_sites_df.iterrows():
    # Determine closest city (simplified: use first city for first site, second for others)
    if index == 0:
        closest_city = cities_df.iloc[0]
    else:
        closest_city = cities_df.iloc[1] if len(cities_df) > 1 else cities_df.iloc[0]
    
    # Calculate distance to the city
    city_distance = calculate_distance(
        launch_site['Latitude'], 
        launch_site['Longitude'],
        closest_city['Latitude'],
        closest_city['Longitude']
    )
    
    # Create a marker at the city location
    marker = folium.Marker(
        location=[closest_city['Latitude'], closest_city['Longitude']],
        icon=DivIcon(
            icon_size=(150, 36),
            icon_anchor=(0, 0),
            html='<div style="font-size: 10pt; color:#e74c3c;"><b>%s (%.2f km)</b></div>' % (closest_city['Name'], city_distance),
        )
    )
    site_map.add_child(marker)
    
    # Draw a line between launch site and city
    coordinates = [
        [launch_site['Latitude'], launch_site['Longitude']], 
        [closest_city['Latitude'], closest_city['Longitude']]
    ]
    lines = folium.PolyLine(locations=coordinates, weight=2, color='red', opacity=0.5)
    site_map.add_child(lines)

site_map

## 9. Geographic Interpretation
Use the map outputs to assess coastline access, distance from populated areas, and potential safety implications for launch operations.

## 10. Summary Statistics
Compute launch-level and site-level summary metrics for final review.

In [18]:
# Calculate summary statistics for the launches
print("="*60)
print("SPACEX LAUNCH ANALYSIS SUMMARY")
print("="*60)

# Overall statistics
total_launches = len(spacex_df)
successful_launches = len(spacex_df[spacex_df['Class'] == 1])
failed_launches = len(spacex_df[spacex_df['Class'] == 0])
success_rate = (successful_launches / total_launches) * 100

print(f"\nOverall Statistics:")
print(f"  Total Launches: {total_launches}")
print(f"  Successful Landings: {successful_launches}")
print(f"  Failed Landings: {failed_launches}")
print(f"  Success Rate: {success_rate:.2f}%")

# Statistics by launch site
print(f"\nLaunch Site Statistics:")
print("-"*60)
for site in spacex_df['LaunchSite'].unique():
    site_data = spacex_df[spacex_df['LaunchSite'] == site]
    site_total = len(site_data)
    site_success = len(site_data[site_data['Class'] == 1])
    site_rate = (site_success / site_total) * 100 if site_total > 0 else 0
    
    print(f"\n{site}:")
    print(f"  Total Launches: {site_total}")
    print(f"  Successful: {site_success}")
    print(f"  Success Rate: {site_rate:.2f}%")

# Geographic information
print(f"\nGeographic Information:")
print("-"*60)
for index, site in launch_sites_df.iterrows():
    print(f"\n{site['LaunchSite']}:")
    print(f"  Latitude: {site['Latitude']:.4f}")
    print(f"  Longitude: {site['Longitude']:.4f}")
    if 'DistanceToCoastline' in site:
        print(f"  Distance to Coastline: {site['DistanceToCoastline']:.2f} km")

print("\n" + "="*60)

SPACEX LAUNCH ANALYSIS SUMMARY

Overall Statistics:
  Total Launches: 90
  Successful Landings: 60
  Failed Landings: 30
  Success Rate: 66.67%

Launch Site Statistics:
------------------------------------------------------------

CCSFS SLC 40:
  Total Launches: 55
  Successful: 33
  Success Rate: 60.00%

VAFB SLC 4E:
  Total Launches: 13
  Successful: 10
  Success Rate: 76.92%

KSC LC 39A:
  Total Launches: 22
  Successful: 17
  Success Rate: 77.27%

Geographic Information:
------------------------------------------------------------

CCSFS SLC 40:
  Latitude: 28.5619
  Longitude: -80.5774
  Distance to Coastline: 1.03 km

KSC LC 39A:
  Latitude: 28.6081
  Longitude: -80.6040
  Distance to Coastline: 6.14 km

VAFB SLC 4E:
  Latitude: 34.6321
  Longitude: -120.6108
  Distance to Coastline: 0.99 km

